# Module 02: Git & Linux Essentials + SQL Foundations

**Program:** Quintrix Jr. Data Engineer Training  
**Duration:** 2 hours  
**Instructor:** Sunil Mogadati

---

**Today's Agenda:**

| # | Topic | Time |
|---|-------|------|
| 1 | Project Structure & `.gitignore` for Data Projects | 15 min |
| 2 | Git Workflow for Data Pipelines | 15 min |
| 3 | Command-Line Data Exploration | 15 min |
| 4 | SQL Foundations — Why SQL for Data Engineers | 10 min |
| 5 | Build the Call Center Database | 20 min |
| 6 | SQL Warm-Up Exercises | 30 min |
| 7 | Key Takeaways | 5 min |
| 8 | Homework | 10 min |

---

## 1. Project Structure & `.gitignore` for Data Projects

**One-line answer:** A data project without structure is a data project that breaks.

Every data engineering project you work on — in this program and in production — follows a standard folder structure. Here's the layout we'll use for the call center analytics project:

```
de-call-center-analytics/
├── data/raw/          # Bronze — never modify source files
├── data/processed/    # Silver/Gold outputs
├── notebooks/         # Exploration, learning
├── src/               # Pipeline code (PySpark transforms, SQL)
├── tests/             # Data quality tests
├── dags/              # Airflow DAGs (Week 3)
├── .gitignore
├── requirements.txt
└── README.md
```

### What Goes in Git vs. What Doesn't

| In Git (track these) | NOT in Git (`.gitignore`) | Why |
|---------------------|--------------------------|-----|
| `src/` — pipeline code | `data/` — CSV, JSON, Parquet files | Data files are too large, may contain PII |
| `dags/` — Airflow DAGs | `.env` — credentials, API keys | Secrets must never be in version control |
| `tests/` — quality checks | `__pycache__/` — Python bytecode | Auto-generated, not source code |
| `requirements.txt` | `.ipynb_checkpoints/` | Jupyter temp files |
| `README.md` | `*.db` — SQLite databases | Generated artifacts, not source |
| `notebooks/` — learning notebooks | `*.log` — log files | Ephemeral, often large |

> **Rule of thumb:** If it's code, config, or documentation — track it. If it's data, secrets, or generated files — ignore it.

In [ ]:
import os

# Standard DE project folder structure
# This matches what every professional DE team uses — not just for this program
project = "de-call-center-analytics"
folders = [
    "data/raw",        # Raw source files — NEVER modify. Bronze layer in medallion architecture.
    "data/processed",  # Cleaned/transformed outputs. Silver/Gold. These are GENERATED — gitignore them.
    "notebooks",       # Exploration and learning notebooks — tracked in Git for sharing and review
    "src",             # Production pipeline code: PySpark transforms, SQL scripts, loaders
    "tests",           # Data quality tests — every pipeline needs them. Catches bugs before prod.
    "dags",            # Airflow DAG definitions — the schedule that orchestrates your pipeline
]

print(f"Project: {project}/")
for folder in folders:
    print(f"  \u251c\u2500\u2500 {folder}/")

# .gitignore tells Git: "skip these files completely"
# WHY: Data files can be GBs in size, contain PII, and change constantly — wrong for version control
# WHY: Credentials (.env, .pem) in Git = immediate security incident. Teams get fired for this.
# WHY: Generated files (__pycache__, *.pyc) clutter diffs and add no value to track
gitignore_content = """# Data files — too large, may contain PII
*.csv
*.json
*.parquet
*.avro
data/

# Credentials — NEVER commit
.env
*.pem
service-account-*.json

# Python artifacts
__pycache__/
*.pyc
.ipynb_checkpoints/

# Generated files
*.db
*.log
.DS_Store
"""

print("\n.gitignore for data projects:")
print("=" * 40)
print(gitignore_content)
# Rule: If it changes every run or contains secrets, it does NOT belong in Git
print("This file goes in the project root. Git will skip everything listed here.")


---

## 2. Git Workflow for Data Pipelines

**One-line answer:** Git isn't just version control — it's your team's communication protocol.

**What is Git?** Think of Git as a time machine + collaboration tool for your code. Every time you save a "snapshot" (called a *commit*), Git records exactly what changed and who changed it. If something breaks, you can rewind to any previous snapshot. If you're working with a team, Git lets everyone work on different parts simultaneously without overwriting each other's work.

### The Feature Branch Workflow

```
main ─────●──────────●──────────●────── (production — always stable)
           \        ↗          ↗
            ●──●──●   ●──●──●
           feature/   feature/
           add-mart   fix-utc-bug
```

- **Never commit directly to `main`** — always use a feature branch
- Create a Pull Request (PR) when your feature is ready
- A teammate reviews your code, then it merges to `main`


**What is a Pull Request (PR)?** A pull request is a formal way of saying: "I finished my work on this branch — can a teammate review it before we merge it into the main codebase?" It's the standard gate for code review on every professional team.

### Branch Naming Conventions

| Branch | Purpose | Example |
|--------|---------|--------|
| `main` | Production-ready code | Always stable, deployable |
| `dev` | Integration branch (some teams) | Merges features before main |
| `feature/...` | New functionality | `feature/add-campaign-mart` |
| `fix/...` | Bug fixes | `fix/utc-date-conversion` |
| `refactor/...` | Code improvement | `refactor/optimize-spark-join` |

### Commit Message Conventions

| Prefix | Meaning | Example |
|--------|---------|--------|
| `feat:` | New feature | `feat: add hourly call volume mart` |
| `fix:` | Bug fix | `fix: correct UTC-to-EST conversion` |
| `refactor:` | Code restructure (no behavior change) | `refactor: split transform into stages` |
| `docs:` | Documentation | `docs: add schema diagram to README` |
| `data:` | Data model change | `data: add channel column to calls table` |
| `test:` | Add or fix tests | `test: add null check for payment amount` |

### Solo Git vs. Team Git

| Aspect | Solo (learning) | Team (production) |
|--------|----------------|------------------|
| Branching | Optional but good practice | Required — never push to main |
| Commit messages | Short is fine | Structured with prefixes |
| Pull requests | Not needed | Required — code review before merge |
| `.gitignore` | Still important | Critical — secrets exposure = incident |
| Frequency | Commit when done | Commit often — small, logical units |

> **In this program:** Every module's work goes into your Git repo. By Week 4, your commit history IS your portfolio evidence.

In [ ]:
# Git commands you'll use daily — run these in your TERMINAL, not here
# This cell shows the workflow; practice it in your local terminal

git_workflow = """
# === Daily Git Workflow for Data Engineers ===

# 1. Start your day — pull latest changes
git pull origin main

# 2. Create a feature branch for today's work
git checkout -b feature/add-hourly-volume-mart

# 3. Do your work... write transforms, tests, SQL...

# 4. Check what changed
git status                              # see modified files
git diff src/transforms/hourly_volume.py  # see exact changes

# 5. Stage specific files (NOT git add . — be intentional)
git add src/transforms/hourly_volume.py
git add tests/test_hourly_volume.py

# 6. Commit with a meaningful message
git commit -m "feat: add hourly call volume mart"

# 7. Push your branch to GitHub
git push -u origin feature/add-hourly-volume-mart

# 8. Create a Pull Request on GitHub (browser or CLI)
# After review and approval, merge to main

# 9. Switch back to main and pull the merged code
git checkout main
git pull origin main
"""

print(git_workflow)
print("Tip: Use 'git add <specific file>' instead of 'git add .'")
print("     This prevents accidentally committing data files or secrets.")

---

### Why Does Version Control Matter for Data Pipelines Specifically?

Version control isn't new — software engineers have used Git for decades. But data engineers have extra reasons to care:

| Scenario | Without Git | With Git |
|----------|-------------|----------|
| Pipeline breaks in production | "Who changed what and when?" — nobody knows | `git log` + `git diff` shows exactly what changed |
| You need to reprocess historical data | "Which version of the transform ran on March 10?" — unknown | Tag the release, check it out, reprocess |
| Two engineers edit the same DAG | One overwrites the other's work | Feature branches + PR prevents silent overwrites |
| Compliance audit asks for change history | No evidence | Full audit trail: who, what, when — tied to every deploy |
| New engineer joins the team | Reads pipeline code, guesses at intent | `git log --oneline` reads like a project diary |

> **In production:** Every pipeline deploy is a Git commit. If the pipeline produces wrong numbers, the first question is always: "What changed?" Git answers it instantly.

**Data engineers also face risks that app developers don't:**
- Large data files accidentally committed — repo balloons to GB size, team can't clone it
- Credentials in a `.env` accidentally staged — `git push` = security incident
- `.ipynb_checkpoints/` clutters every diff with JSON noise on every save

The `.gitignore` from Section 1 protects against all three.

---

### Git Gotchas for Data Engineers

These mistakes happen to beginners and experienced engineers alike. Memorize them.

**Gotcha 1: Committing a `.env` file**
```bash
# You do this...
git add .
git commit -m "feat: add database connection"
git push

# Now your DB password is on GitHub — even if you delete the file in the next commit,
# it is still in git history. Fix requires git filter-branch or BFG Repo Cleaner (painful).
# Prevention: .gitignore + always use 'git add <specific file>' not 'git add .'
```

**Gotcha 2: Committing a large data file**
```bash
git add data/raw/big_file.parquet   # 500 MB accidentally staged
git commit -m "add raw data"
git push

# Now everyone who clones the repo downloads 500 MB.
# Removing large files from history is painful.
# Prevention: data/ in .gitignore. Check 'git status' before staging.
```

**Gotcha 3: Pushing directly to main**
```bash
git push origin main   # broken transform goes straight to production

# Prevention: Protect the main branch on GitHub
# Settings -> Branches -> Branch protection rules -> Require pull request before merging
```

**Gotcha 4: Committing notebook checkpoints**
```bash
# Jupyter auto-creates .ipynb_checkpoints/ on every save
# Without .gitignore, these appear in every diff as meaningless JSON changes
# Prevention: Add .ipynb_checkpoints/ to .gitignore before your first commit
```

**Gotcha 5: Vague commit messages**
```bash
git commit -m "fix stuff"                                                     # useless in code review or audit
git commit -m "fix: correct UTC-to-EST offset in daily_calls_mart.py"       # useful
```

> **Interview tip:** If asked "how do you handle secrets in a pipeline?" — mention `.env` files, `.gitignore`, and secrets managers (AWS Secrets Manager, GCP Secret Manager). Never hardcode. Never commit.

---

## 3. Command-Line Data Exploration

**One-line answer:** Before you write Python, profile your data from the command line in 10 seconds.


Think of the terminal as a text-based remote control for your computer — instead of clicking icons, you type instructions. It feels unfamiliar at first, but it's faster and more powerful for data work.

> **Windows users:** These commands work on Mac and Linux. On Windows, use **Git Bash** (installed with Git) or **WSL** (Windows Subsystem for Linux). Setup instructions are in M00.



### Essential Commands for Data Engineers

| Command | What It Does | DE Example |
|---------|-------------|------------|
| `head -n 5 file.csv` | Show first 5 lines | Check the header and first few records |
| `tail -n 5 file.csv` | Show last 5 lines | Check if the file ended cleanly |
| `wc -l file.csv` | Count lines | How many records? (subtract 1 for header) |
| `grep "dropped" file.csv` | Find lines containing a string | How many dropped calls? |
| `grep -c "dropped" file.csv` | Count matching lines | Get the count directly |
| `cut -d',' -f5 file.csv` | Extract column 5 (comma-delimited) | Pull out just the disposition column |
| `sort` | Sort lines alphabetically | Prepare for `uniq` |
| `uniq -c` | Count consecutive duplicates | Frequency counts (must sort first) |
| `\|` (pipe) | Send output of one command to another | Chain commands together |
| `cat file.json \| jq '.'` | Pretty-print JSON | Inspect raw API responses |

> **Why this matters:** When a 2 GB file lands in your pipeline and something looks wrong, you don't want to wait for Python to load it. These commands give you instant answers.

In [ ]:
# Create a sample call center CSV to explore from the command line
# WHY CSV? It's the most common raw ingest format — logs, exports, API responses all land as CSV first.
# This simulates what lands in data/raw/ at the start of a real pipeline.

# Column breakdown:
#   call_id         — unique identifier for each call (like a receipt number)
#   dnis            — the phone number the customer DIALED (tells us which campaign/client)
#   caller_ani      — the customer's phone number (Automatic Number Identification)
#   call_started_at — UTC timestamp (always store UTC; convert to local time for display)
#   disposition     — outcome: completed = agent helped, dropped = hung up, transferred = routed elsewhere
#   amount          — revenue collected on this call (0.00 for non-sales outcomes like drops/transfers)
#
# WHY intentional $0.00 on dropped/transferred calls?
#   This lets us practice: WHERE amount > 0 vs WHERE disposition = 'completed'
#   (Spoiler: they're not always the same — a transferred call CAN convert downstream)

sample_data = """call_id,dnis,caller_ani,call_started_at,disposition,amount
call_001,8005551234,3135559876,2026-03-10T19:30:00Z,completed,79.99
call_002,8005551234,2485555678,2026-03-10T20:15:00Z,dropped,0.00
call_003,8005552345,7345551234,2026-03-10T21:00:00Z,completed,119.98
call_004,8005552345,3135554321,2026-03-10T22:30:00Z,transferred,0.00
call_005,8005551234,2485559999,2026-03-11T00:15:00Z,completed,49.95
call_006,8005553456,5865551111,2026-03-11T01:30:00Z,dropped,0.00
call_007,8005553456,3135558888,2026-03-11T14:00:00Z,completed,79.99
call_008,8005551234,7345557777,2026-03-11T15:45:00Z,completed,159.97
call_009,8005552345,2485556666,2026-03-11T17:00:00Z,dropped,0.00
call_010,8005553456,3135553333,2026-03-11T18:30:00Z,completed,49.95
"""

# Write the CSV to a file — 'w' mode creates or overwrites the file
with open("sample_calls.csv", "w") as f:
    f.write(sample_data.strip())

print("Created sample_calls.csv with 10 call records.")
print("Now let's explore it from the command line...")


In [ ]:
!echo "=== Row count (including header) ==="
!wc -l sample_calls.csv

!echo "\n=== First 4 lines (header + 3 rows) ==="
!head -4 sample_calls.csv

!echo "\n=== Completed calls only ==="
!grep "completed" sample_calls.csv

!echo "\n=== Count by disposition ==="
!tail -n +2 sample_calls.csv | cut -d',' -f5 | sort | uniq -c | sort -rn

!echo "\n=== Count by DNIS (phone number = campaign) ==="
!tail -n +2 sample_calls.csv | cut -d',' -f2 | sort | uniq -c | sort -rn

!echo "\n=== Calls with amount > 0 (revenue-generating) ==="
# awk breakdown: -F',' = use comma as separator, NR>1 = skip header row,
# $6 > 0 = where column 6 (revenue) is positive, print columns 1, 5, 6
!awk -F',' 'NR>1 && $6 > 0 {print $1, $5, "$"$6}' sample_calls.csv

### Environment Variables & Bash Scripts

Two more Linux concepts that come up constantly in DE:


Think of environment variables as sticky notes on your computer's desk — programs can read them without you having to hardcode values into your code.

**Environment variables** store configuration outside your code:

```bash
# Set variables (in terminal or .env file)
export DB_HOST=localhost
export DB_PASSWORD=secret123

# Access in Python
import os
host = os.environ.get("DB_HOST")
```

**Bash scripts** automate repetitive tasks:

```bash
#!/bin/bash
# run_pipeline.sh — daily pipeline trigger
echo "Starting pipeline at $(date)"
python src/extract.py
python src/transform.py
python src/load.py
echo "Pipeline complete at $(date)"
```

| Concept | Why DEs Care |
|---------|-------------|
| Environment variables | Database credentials, API keys, cloud project IDs — never hardcode |
| `.env` files | Store env vars locally; `.gitignore` keeps them out of Git |
| Bash scripts | Pipeline entrypoints, cron jobs, Docker entrypoints |
| `chmod +x script.sh` | Make a script executable |
| `cron` / scheduled jobs | Run scripts on a schedule (precursor to Airflow) |

> **In this program:** You'll use `.env` files for database credentials starting in M08 (GCP). Airflow (M11) replaces bash scripts + cron with something much more powerful.

---

## 4. SQL Foundations — Why SQL for Data Engineers

**One-line answer:** SQL is the lingua franca of data. Every tool you'll learn speaks it.

### SQL Is Everywhere in the DE Stack

| Tool | How It Uses SQL |
|------|----------------|
| **BigQuery** | Pure SQL — all queries, transformations, DDL |
| **PySpark** | Spark SQL API: `spark.sql("SELECT * FROM ...")` |
| **Hive** | HiveQL — SQL dialect for Hadoop |
| **Airflow** | SQL operators execute queries as pipeline steps |
| **dbt** | Entire tool is SQL templates + version control |
| **Delta Lake** | SQL reads/writes via Spark SQL |

### Python vs. SQL for Data Operations

| Operation | SQL | Python |
|-----------|-----|--------|
| Filter rows | `WHERE disposition = 'completed'` | `df[df['disposition'] == 'completed']` |
| Aggregate | `GROUP BY campaign, SUM(amount)` | `df.groupby('campaign')['amount'].sum()` |
| Join tables | `JOIN orders ON calls.id = orders.call_id` | `pd.merge(calls, orders, on='call_id')` |
| Window function | `ROW_NUMBER() OVER (PARTITION BY ...)` | Much harder in pandas |
| Run at scale | BigQuery handles petabytes natively | Pandas runs out of memory |

> **Key insight:** You'll write more SQL than Python in most DE jobs. PySpark has a SQL API. BigQuery is pure SQL. Airflow operators execute SQL. SQL is not optional — it's the primary language.

### What We Cover Here vs. M03

| This Module (warm-up) | M03 (Advanced SQL, 5 hours) |
|----------------------|----------------------------|
| SELECT, WHERE, ORDER BY | Window functions (ROW_NUMBER, RANK, LAG/LEAD) |
| GROUP BY, COUNT, SUM | CTEs (Common Table Expressions) |
| INNER JOIN, LEFT JOIN | Subqueries and correlated subqueries |
| Basic date handling | Query optimization and EXPLAIN plans |
| Schema understanding | SCD Type 2 and slowly changing dimensions |

---

## 5. Build the Call Center Database

Let's build a simplified version of a real call center database. The platform this is modeled on is primarily a **live agent** operation — human agents handling inbound sales and service calls. Recently, 3 of its client programs added **AI-powered virtual agents (VA)** using RetellAI, so the system now handles both channels. All data below is synthetic — no production data is used.

### Schema Diagram

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│    calls     │     │    orders     │     │   payments   │
├──────────────┤     ├──────────────┤     ├──────────────┤
│ call_id (PK) │──┐  │ order_id (PK)│──┐  │ txn_id (PK)  │
│ dnis         │  └─→│ call_id (FK) │  └─→│ order_id (FK)│
│ caller_ani   │     │ sku          │     │ auth_code    │
│ call_started │     │ subtotal     │     │ amount       │
│ call_ended   │     │ tax          │     │ status       │
│ disposition  │     │ shipping     │     └──────────────┘
│ sentiment    │     │ total        │
│ channel      │     └──────────────┘
└──────┬───────┘
       │
       │ dnis
       ▼
┌──────────────┐     ┌──────────────┐
│   sources    │     │   clients    │
├──────────────┤     ├──────────────┤
│ dnis (PK)    │     │ client_id(PK)│
│ campaign_name│     │ client_name  │
│ client_name  │────→│ industry     │
│ channel      │     └──────────────┘
└──────────────┘
```

**PK** = Primary Key: a unique identifier for each row (no duplicates allowed).  
**FK** = Foreign Key: a link to a row in another table — this is how tables connect to each other.

Think of tables like separate Excel sheets, and foreign keys as the column that links one sheet to another — like having a `call_id` column in both the Calls sheet and the Orders sheet so you can match them up.

| Table | Purpose | Row Count |
|-------|---------|----------|
| `calls` | Every inbound call — live agent and VA | ~40 |
| `orders` | Orders placed during or after a call | ~15 |
| `payments` | Payment gateway responses | ~15 |
| `sources` | DNIS (phone number) → campaign + channel mapping | 6 |
| `clients` | Client master data | 3 |

> **Note:** The `channel` column distinguishes live agent calls from VA calls. Most calls are live agent — VA is the newer addition for 3 programs. You'll use this column in M03 to compare performance across channels.

You'll use these same tables through M03 (Advanced SQL), M06 (PySpark), and M16 (Capstone).

---

### Hello World: A Database in 10 Lines

Before building the full 5-table call center schema, let's prove the concept with the smallest possible example. Run this cell first — it makes everything that follows click faster.

In [ ]:
# Hello World: The simplest possible database
# Before we build a 5-table schema, let's prove the concept with the absolute minimum.
# This is all a database is: tables with rows. SQL asks questions about those rows.

import sqlite3

# in-memory database — disappears when this connection closes, no cleanup needed
hw_conn = sqlite3.connect(":memory:")
hw_cursor = hw_conn.cursor()

# CREATE TABLE defines the structure — column names and data types
# TEXT = string, INTEGER = whole number, REAL = decimal (floating point)
hw_cursor.execute("""
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,   -- unique row identifier — no two rows can share this value
    product_name TEXT NOT NULL,        -- every product must have a name (NOT NULL enforces this rule)
    price REAL                         -- price can be NULL if not yet set
)
""")

# INSERT adds rows — one row per VALUES() call
# The order of values must match the column order in CREATE TABLE
hw_cursor.execute("INSERT INTO products VALUES (1, 'Widget A', 9.99)")
hw_cursor.execute("INSERT INTO products VALUES (2, 'Widget B', 14.99)")

# SELECT retrieves rows — * means all columns
# fetchall() returns a Python list of tuples — each tuple is one database row
rows = hw_cursor.execute("SELECT * FROM products").fetchall()

print("=== Our tiny database ===")
for row in rows:
    print(f"  ID: {row[0]}, Name: {row[1]}, Price: ${row[2]:.2f}")

# Ask a question with WHERE and ORDER BY
# fetchone() returns only the first result row (we expect just one here)
result = hw_cursor.execute(
    "SELECT product_name, price FROM products WHERE price > 10 ORDER BY price DESC"
).fetchone()
print(f"\nProduct over $10: {result[0]} (${result[1]:.2f})")

# Close this connection — the main schema below uses its own separate connection (conn/cursor)
hw_conn.close()

print("\n--- That is it. That is a database. ---")
print("Now we will scale this exact pattern to 5 tables and 40 calls.")
print("Every concept below — CREATE, INSERT, SELECT, JOIN — builds on exactly this.")


In [ ]:
import sqlite3

# ":memory:" = creates the database in RAM, not on disk
# WHY in-memory? No file cleanup needed, instant startup, perfect for learning and testing
# In production you'd use: conn = sqlite3.connect("call_center.db") or a cloud database URL
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()  # cursor = the "pen" you use to write SQL commands to the database

# Table 1: calls — every inbound call (live agent and VA)
# Central fact table — orders, payments, and sources all link back to calls
cursor.execute("""
CREATE TABLE calls (
    call_id TEXT PRIMARY KEY,         -- Unique ID for every call. TEXT because IDs are strings like "call_001"
    dnis TEXT NOT NULL,               -- Phone number dialed. Links to sources table to get campaign info
    caller_ani TEXT,                   -- Customer's phone number. NULL allowed — blocked/private numbers show nothing
    call_started_at TEXT NOT NULL,    -- UTC timestamp. NOT NULL — if we don't know when it started, the row is useless
    call_ended_at TEXT,               -- UTC timestamp. NULL = call still in progress or duration unknown
    disposition TEXT,                  -- Outcome: completed | dropped | transferred. NULL = system error
    sentiment TEXT,                    -- AI-scored tone: positive | neutral | negative. May be NULL for dropped calls
    channel TEXT NOT NULL              -- live_agent = human agent handled it, va = virtual agent (RetellAI)
)
""")

# Table 2: orders — placed during or after a call
# WHY separate from calls? Not every call results in an order.
# LEFT JOIN reveals the many calls with NO order (dropped, negative sentiment, etc.)
cursor.execute("""
CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    call_id TEXT,                      -- FK -> calls.call_id. Links back to the call that created this order
    sku TEXT NOT NULL,                 -- Stock Keeping Unit — the product code. "SKU-AC-1001" = Acme product #1001
    subtotal REAL,                     -- Price before tax/shipping. REAL = floating point (decimals allowed)
    tax REAL,                          -- Tax amount in dollars (not rate). Already calculated.
    shipping REAL,                     -- Shipping charge. Can be 0.00 for free shipping promotions
    total REAL,                        -- subtotal + tax + shipping. Stored to avoid re-adding every query
    FOREIGN KEY (call_id) REFERENCES calls(call_id)   -- Enforces: no orphaned orders without a parent call
)
""")

# Table 3: payments — payment gateway responses
# One order -> one payment attempt (simplified model)
# In real systems there can be retries, partial payments, refunds — each gets its own row
cursor.execute("""
CREATE TABLE payments (
    transaction_id TEXT PRIMARY KEY,
    order_id TEXT,                     -- FK -> orders.order_id. Links payment back to what was purchased
    auth_code TEXT,                    -- Authorization code from payment processor. NULL if declined (no approval)
    amount REAL,                       -- Amount charged. Should match orders.total — mismatches = reconciliation bug
    status TEXT,                       -- approved | declined | pending. Declined rows will have NULL auth_code
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
)
""")

# Table 4: sources — DNIS to campaign + channel mapping
# WHY separate? A DNIS can be reassigned to a new campaign. Store the mapping once,
# update it in one place, and all queries automatically reflect the change
cursor.execute("""
CREATE TABLE sources (
    dnis TEXT PRIMARY KEY,             -- The phone number. PK = each number appears exactly once
    campaign_name TEXT NOT NULL,       -- Human-readable campaign label for reporting
    client_name TEXT NOT NULL,         -- Which client owns this campaign
    channel TEXT NOT NULL              -- live_agent or va — set at the campaign level, not per-call
)
""")

# Table 5: clients — client master data
# Rarely changes — maybe a few rows added per year
# WHY INTEGER PRIMARY KEY here? Numeric IDs are faster for joins and common in enterprise systems
cursor.execute("""
CREATE TABLE clients (
    client_id INTEGER PRIMARY KEY,
    client_name TEXT NOT NULL,
    industry TEXT                      -- NULL allowed — not all clients report industry in onboarding
)
""")

print("5 tables created: calls, orders, payments, sources, clients")
print("\nThis mirrors a real production call center database.")
print("The platform handles both live agent and VA (virtual agent) calls.")
print("All data is synthetic — no production data is used.")


In [ ]:
# === Insert sample data ===
# executemany() runs the same INSERT for every tuple in the list — one DB round-trip per batch
# Much faster than calling cursor.execute() in a for loop for large datasets

# --- Clients (3 clients with industries) ---
# Three fictional clients representing three verticals: Retail, Healthcare, Financial Services
# WHY three? Enough variety to practice GROUP BY client, not so many the data gets confusing
clients_data = [
    (1, "Acme Corp", "Retail"),
    (2, "Pinnacle Health", "Healthcare"),
    (3, "Summit Financial", "Financial Services"),
]
cursor.executemany("INSERT INTO clients VALUES (?, ?, ?)", clients_data)

# --- Sources — 6 DNIS entries (2 campaigns per client) ---
# Each client has 2 campaigns: one live_agent, one va
# WHY this split? Mirrors reality — VA is newer, added alongside existing live-agent programs
# The DNIS is the phone number customers dial. It determines which campaign/channel the call belongs to.
# Pattern: 800-555-1234 = Acme live agent, 800-555-1235 = Acme VA (consecutive numbers = same client)
sources_data = [
    ("8005551234", "Acme Spring Sale",        "Acme Corp",          "live_agent"),
    ("8005551235", "Acme Rewards Program",    "Acme Corp",          "va"),
    ("8005552345", "Pinnacle Wellness Plan",  "Pinnacle Health",    "live_agent"),
    ("8005552346", "Pinnacle Rx Refill",      "Pinnacle Health",    "va"),
    ("8005553456", "Summit Auto Loan",        "Summit Financial",   "live_agent"),
    ("8005553457", "Summit Credit Card",      "Summit Financial",   "va"),
]
cursor.executemany("INSERT INTO sources VALUES (?, ?, ?, ?)", sources_data)

# --- Calls — 40 rows spanning 2 calendar days ---
# WHY 40 calls? Small enough to read, large enough to see patterns in GROUP BY aggregations
# Disposition mix: ~60% completed (24), ~25% dropped (10), ~15% transferred (6)
# All timestamps are UTC (Z suffix) — intentional, sets up the UTC bug in Exercise 4
# Channel is determined by DNIS: x1234/x2345/x3456 = live_agent, x1235/x2346/x3457 = va
calls_data = [
    # Day 1, March 10 — Daytime EST (8 AM-12 PM EST = 13:00-17:00 UTC)
    ("call_001", "8005551234", "3135559876", "2026-03-10T13:00:00Z", "2026-03-10T13:07:45Z", "completed",   "positive", "live_agent"),
    ("call_002", "8005551234", "2485555678", "2026-03-10T13:30:00Z", "2026-03-10T13:32:15Z", "dropped",     "negative", "live_agent"),
    ("call_003", "8005552345", "7345551234", "2026-03-10T14:00:00Z", "2026-03-10T14:12:30Z", "completed",   "neutral",  "live_agent"),
    ("call_004", "8005552346", "3135554321", "2026-03-10T14:30:00Z", "2026-03-10T14:35:00Z", "transferred", "neutral",  "va"),
    ("call_005", "8005553456", "2485559999", "2026-03-10T15:00:00Z", "2026-03-10T15:08:20Z", "completed",   "positive", "live_agent"),
    ("call_006", "8005551235", "5865551111", "2026-03-10T15:30:00Z", "2026-03-10T15:31:10Z", "dropped",     "negative", "va"),
    ("call_007", "8005553457", "3135558888", "2026-03-10T16:00:00Z", "2026-03-10T16:06:45Z", "completed",   "positive", "va"),
    ("call_008", "8005551234", "7345557777", "2026-03-10T16:30:00Z", "2026-03-10T16:42:00Z", "completed",   "neutral",  "live_agent"),
    ("call_009", "8005552345", "2485556666", "2026-03-10T17:00:00Z", "2026-03-10T17:01:30Z", "dropped",     "negative", "live_agent"),
    ("call_010", "8005553456", "3135553333", "2026-03-10T17:30:00Z", "2026-03-10T17:38:15Z", "completed",   "positive", "live_agent"),
    # Day 1 afternoon/evening EST (1 PM-5 PM EST = 18:00-22:00 UTC)
    ("call_011", "8005551234", "5865552222", "2026-03-10T18:00:00Z", "2026-03-10T18:09:30Z", "completed",   "neutral",  "live_agent"),
    ("call_012", "8005552346", "7345554444", "2026-03-10T18:30:00Z", "2026-03-10T18:33:00Z", "transferred", "neutral",  "va"),
    ("call_013", "8005551235", "3135551111", "2026-03-10T19:00:00Z", "2026-03-10T19:05:45Z", "completed",   "positive", "va"),
    ("call_014", "8005553457", "2485553333", "2026-03-10T19:30:00Z", "2026-03-10T19:31:00Z", "dropped",     "negative", "va"),
    ("call_015", "8005551234", "5865554444", "2026-03-10T20:00:00Z", "2026-03-10T20:11:20Z", "completed",   "positive", "live_agent"),
    ("call_016", "8005552345", "7345558888", "2026-03-10T20:30:00Z", "2026-03-10T20:36:00Z", "completed",   "neutral",  "live_agent"),
    ("call_017", "8005553456", "3135559999", "2026-03-10T21:00:00Z", "2026-03-10T21:02:00Z", "dropped",     "negative", "live_agent"),
    ("call_018", "8005551235", "2485551111", "2026-03-10T21:30:00Z", "2026-03-10T21:40:15Z", "completed",   "neutral",  "va"),
    ("call_019", "8005552346", "5865557777", "2026-03-10T22:00:00Z", "2026-03-10T22:04:30Z", "transferred", "neutral",  "va"),
    ("call_020", "8005553457", "7345552222", "2026-03-10T22:30:00Z", "2026-03-10T22:37:45Z", "completed",   "positive", "va"),
    # THE UTC BUG ZONE: 7 PM-9 PM EST = next calendar day in UTC
    # These 5 calls happened on the EVENING of March 10 (EST) but UTC rolls to March 11
    # A naive DATE() on call_started_at will count them as March 11 — wrong daily totals!
    ("call_021", "8005551234", "3135556666", "2026-03-11T00:00:00Z", "2026-03-11T00:09:30Z", "completed",   "neutral",  "live_agent"),  # 7:00 PM EST Mar 10
    ("call_022", "8005552345", "2485558888", "2026-03-11T00:30:00Z", "2026-03-11T00:31:45Z", "dropped",     "negative", "live_agent"),  # 7:30 PM EST Mar 10
    ("call_023", "8005553456", "5865553333", "2026-03-11T01:00:00Z", "2026-03-11T01:06:00Z", "completed",   "positive", "live_agent"),  # 8:00 PM EST Mar 10
    ("call_024", "8005551235", "7345559999", "2026-03-11T01:30:00Z", "2026-03-11T01:38:20Z", "completed",   "positive", "va"),          # 8:30 PM EST Mar 10
    ("call_025", "8005553457", "3135552222", "2026-03-11T02:00:00Z", "2026-03-11T02:03:15Z", "transferred", "neutral",  "va"),          # 9:00 PM EST Mar 10
    # Day 2, March 11 — Proper daytime EST calls (13:00-20:00 UTC)
    ("call_026", "8005551234", "2485554444", "2026-03-11T13:00:00Z", "2026-03-11T13:05:30Z", "completed",   "positive", "live_agent"),
    ("call_027", "8005552345", "5865556666", "2026-03-11T13:30:00Z", "2026-03-11T13:32:00Z", "dropped",     "negative", "live_agent"),
    ("call_028", "8005553456", "7345553333", "2026-03-11T14:00:00Z", "2026-03-11T14:10:45Z", "completed",   "neutral",  "live_agent"),
    ("call_029", "8005551235", "3135557777", "2026-03-11T14:30:00Z", "2026-03-11T14:34:00Z", "transferred", "neutral",  "va"),
    ("call_030", "8005552346", "2485552222", "2026-03-11T15:00:00Z", "2026-03-11T15:08:30Z", "completed",   "positive", "va"),
    ("call_031", "8005553457", "5865559999", "2026-03-11T15:30:00Z", "2026-03-11T15:31:20Z", "dropped",     "negative", "va"),
    ("call_032", "8005551234", "7345556666", "2026-03-11T16:00:00Z", "2026-03-11T16:07:00Z", "completed",   "neutral",  "live_agent"),
    ("call_033", "8005552345", "3135554444", "2026-03-11T16:30:00Z", "2026-03-11T16:41:30Z", "completed",   "positive", "live_agent"),
    ("call_034", "8005553456", "2485557777", "2026-03-11T17:00:00Z", "2026-03-11T17:01:45Z", "dropped",     "negative", "live_agent"),
    ("call_035", "8005551235", "5865558888", "2026-03-11T17:30:00Z", "2026-03-11T17:36:15Z", "completed",   "positive", "va"),
    ("call_036", "8005552346", "7345551111", "2026-03-11T18:00:00Z", "2026-03-11T18:03:30Z", "transferred", "neutral",  "va"),
    ("call_037", "8005553457", "3135558888", "2026-03-11T18:30:00Z", "2026-03-11T18:39:45Z", "completed",   "neutral",  "va"),
    ("call_038", "8005551234", "2485559876", "2026-03-11T19:00:00Z", "2026-03-11T19:06:20Z", "completed",   "positive", "live_agent"),
    ("call_039", "8005552345", "5865554321", "2026-03-11T19:30:00Z", "2026-03-11T19:32:00Z", "dropped",     "negative", "live_agent"),
    ("call_040", "8005553456", "7345559876", "2026-03-11T20:00:00Z", "2026-03-11T20:07:30Z", "completed",   "positive", "live_agent"),
]
cursor.executemany("INSERT INTO calls VALUES (?, ?, ?, ?, ?, ?, ?, ?)", calls_data)

# --- Orders — 15 rows linked to completed calls ---
# WHY only 15 orders for 40 calls? ~37.5% conversion rate — realistic for inbound sales
# Orders only exist for "completed" calls — dropped/transferred rarely convert in this model
# SKU naming: SKU-{client initials}-{number} (AC=Acme, PH=Pinnacle Health, SF=Summit Financial)
# subtotal + tax + shipping = total (pre-calculated and stored to speed up aggregation queries)
orders_data = [
    ("ord_001", "call_001", "SKU-AC-1001", 69.99,  5.60,  4.40,  79.99),
    ("ord_002", "call_003", "SKU-PH-2001", 99.99,  8.00,  11.99, 119.98),
    ("ord_003", "call_005", "SKU-SF-3001", 39.95,  3.20,  6.80,  49.95),
    ("ord_004", "call_008", "SKU-AC-1002", 139.97, 11.20, 8.80,  159.97),
    ("ord_005", "call_010", "SKU-SF-3001", 39.95,  3.20,  6.80,  49.95),
    ("ord_006", "call_011", "SKU-AC-1001", 69.99,  5.60,  4.40,  79.99),
    ("ord_007", "call_015", "SKU-AC-1003", 89.99,  7.20,  5.80,  102.99),
    ("ord_008", "call_016", "SKU-PH-2002", 59.95,  4.80,  5.20,  69.95),
    ("ord_009", "call_020", "SKU-SF-3002", 149.99, 12.00, 7.00,  168.99),
    ("ord_010", "call_021", "SKU-AC-1001", 69.99,  5.60,  4.40,  79.99),
    ("ord_011", "call_023", "SKU-SF-3001", 39.95,  3.20,  6.80,  49.95),
    ("ord_012", "call_026", "SKU-AC-1002", 139.97, 11.20, 8.80,  159.97),
    ("ord_013", "call_030", "SKU-PH-2001", 99.99,  8.00,  11.99, 119.98),
    ("ord_014", "call_033", "SKU-PH-2002", 59.95,  4.80,  5.20,  69.95),
    ("ord_015", "call_040", "SKU-SF-3002", 149.99, 12.00, 7.00,  168.99),
]
cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)", orders_data)

# --- Payments — 15 rows (13 approved, 2 intentionally declined) ---
# WHY 2 declined? Realistic ~13% payment failure rate; sets up data quality exercises
# Declined rows have NULL auth_code — processor sends no authorization code when it rejects the card
# This NULL is intentional: you will use IS NULL in queries to find declined payments
# txn_006 (ord_006) and txn_012 (ord_012) are the declined ones — note them for M03 exercises
payments_data = [
    ("txn_001", "ord_001", "AUTH-7821", 79.99,  "approved"),
    ("txn_002", "ord_002", "AUTH-7822", 119.98, "approved"),
    ("txn_003", "ord_003", "AUTH-7823", 49.95,  "approved"),
    ("txn_004", "ord_004", "AUTH-7824", 159.97, "approved"),
    ("txn_005", "ord_005", "AUTH-7825", 49.95,  "approved"),
    ("txn_006", "ord_006", None,        79.99,  "declined"),   # NULL auth_code = card declined, no approval issued
    ("txn_007", "ord_007", "AUTH-7827", 102.99, "approved"),
    ("txn_008", "ord_008", "AUTH-7828", 69.95,  "approved"),
    ("txn_009", "ord_009", "AUTH-7829", 168.99, "approved"),
    ("txn_010", "ord_010", "AUTH-7830", 79.99,  "approved"),
    ("txn_011", "ord_011", "AUTH-7831", 49.95,  "approved"),
    ("txn_012", "ord_012", None,        159.97, "declined"),   # NULL auth_code = card declined, no approval issued
    ("txn_013", "ord_013", "AUTH-7833", 119.98, "approved"),
    ("txn_014", "ord_014", "AUTH-7834", 69.95,  "approved"),
    ("txn_015", "ord_015", "AUTH-7835", 168.99, "approved"),
]
cursor.executemany("INSERT INTO payments VALUES (?, ?, ?, ?, ?)", payments_data)

# commit() makes all INSERT changes permanent in the database session
# Without commit(), changes exist only in a transaction buffer and will be LOST if the connection closes
conn.commit()

# Print summary to verify everything loaded correctly
print("=== Data Loaded ===")
tables = ['clients', 'sources', 'calls', 'orders', 'payments']
for table in tables:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count} rows")

# Verify disposition distribution — should be ~60% completed, ~25% dropped, ~15% transferred
print("\n=== Call Disposition Distribution ===")
for row in cursor.execute("SELECT disposition, COUNT(*) as cnt FROM calls GROUP BY disposition ORDER BY cnt DESC"):
    print(f"  {row[0]}: {row[1]}")

# Verify channel distribution — should be 23 live_agent, 17 va
print("\n=== Channel Distribution ===")
for row in cursor.execute("SELECT channel, COUNT(*) as cnt FROM calls GROUP BY channel ORDER BY cnt DESC"):
    print(f"  {row[0]}: {row[1]}")


In [ ]:
# Quick data profile — what do we have?

print("=== Data Profile ===")
print()

# Table row counts
tables = ['calls', 'orders', 'payments', 'sources', 'clients']
for table in tables:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count} rows")

# Date range
min_date = cursor.execute("SELECT MIN(call_started_at) FROM calls").fetchone()[0]
max_date = cursor.execute("SELECT MAX(call_started_at) FROM calls").fetchone()[0]
print(f"\n  Call date range (UTC): {min_date} to {max_date}")

# Unique DNIS count
dnis_count = cursor.execute("SELECT COUNT(DISTINCT dnis) FROM calls").fetchone()[0]
print(f"  Unique DNIS numbers: {dnis_count}")

# Channel breakdown
print("\n  Channel breakdown:")
for row in cursor.execute("SELECT channel, COUNT(*) FROM calls GROUP BY channel ORDER BY COUNT(*) DESC"):
    print(f"    {row[0]}: {row[1]} calls")

# Conversion rate
total_calls = cursor.execute("SELECT COUNT(*) FROM calls").fetchone()[0]
calls_with_orders = cursor.execute("SELECT COUNT(DISTINCT call_id) FROM orders").fetchone()[0]
print(f"\n  Conversion: {calls_with_orders}/{total_calls} calls resulted in orders ({calls_with_orders/total_calls*100:.1f}%)")

# Payment success rate
total_payments = cursor.execute("SELECT COUNT(*) FROM payments").fetchone()[0]
approved = cursor.execute("SELECT COUNT(*) FROM payments WHERE status = 'approved'").fetchone()[0]
print(f"  Payment approval: {approved}/{total_payments} approved ({approved/total_payments*100:.1f}%)")

---

## 6. SQL Warm-Up Exercises

Four progressive exercises using the call center data. Each builds on the previous one.

---

### Exercise 1: SELECT + WHERE + ORDER BY

**Task:** Show all completed calls, sorted by start time.

Key SQL concepts:
- `SELECT` chooses which columns to return
- `WHERE` filters rows
- `ORDER BY` sorts results (default ascending, `DESC` for descending)

---

### Try It Yourself — Exercise 1

Before looking at the solution below, write this query yourself.

**Task:** Show all *dropped* calls, sorted by start time newest-first.

```sql
SELECT call_id, dnis, channel, call_started_at, sentiment
FROM calls
WHERE ___________
ORDER BY ___________ DESC
```

**Bonus challenge:** Add `AND channel = 'va'` to the WHERE clause. How many VA calls were dropped?

In [ ]:
# Exercise 1: Basic SELECT + WHERE + ORDER BY
# WHY this pattern matters: Every pipeline starts with "show me rows that meet a condition"
# SELECT controls WHICH columns you see. * means all columns, but naming them is better practice.
# WHERE filters ROWS — like Excel's AutoFilter, but composable with AND/OR/NOT
# ORDER BY sorts results — default is ascending (A-Z, oldest-newest); add DESC to flip it

print("=== Completed Calls (sorted by start time) ===")
print(f"{'Call ID':<12} {'DNIS':<12} {'Channel':<12} {'Started At (UTC)':<24} {'Sentiment':<10}")
print("-" * 70)

results = cursor.execute("""
    SELECT call_id, dnis, channel, call_started_at, sentiment
    FROM calls
    WHERE disposition = 'completed'
    ORDER BY call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<12} {row[3]:<24} {row[4]:<10}")

# You should see: 24 completed calls, sorted oldest-first by UTC timestamp
print(f"\nTotal completed calls: {len(results)}")
print("\nNotice: We used WHERE to filter, ORDER BY to sort.")
# BEGINNER NOTE: SQL processes WHERE before ORDER BY — filter first, then sort the filtered set
print("Try changing 'completed' to 'dropped' — how many dropped calls are there?")
print("Try adding: WHERE disposition = 'completed' AND channel = 'va'")


---

### Exercise 2: GROUP BY + COUNT + SUM

**Task:** How many calls per campaign? What's the total revenue per campaign?

Key SQL concepts:
- `GROUP BY` groups rows by one or more columns
- `COUNT(*)` counts rows in each group
- `SUM()` adds values within each group
- `ROUND()` controls decimal places
- `JOIN` connects tables using a shared column

---

### Try It Yourself — Exercise 2

Before looking at the solution, think through the query structure:
1. Which tables do you need to count calls per campaign?
2. Which table has `campaign_name`? Which table has the rows to count?
3. What column links them?

Write the skeleton, then fill in the blanks:
```sql
SELECT s.campaign_name, s.channel, COUNT(*) as call_count
FROM _____ c
JOIN _____ s ON c._____ = s._____
GROUP BY _____
ORDER BY call_count DESC
```

**Bonus challenge:** Add a WHERE clause to show only `live_agent` campaigns.

In [ ]:
# Exercise 2: Aggregations — GROUP BY + COUNT + SUM with JOIN
# WHY this pattern matters: Aggregation is the most common DE task — "summarize X by Y"
# GROUP BY collapses all rows with the same value into ONE summary row per group
# COUNT(*) counts every row in the group. COUNT(column) counts only non-NULL values — different!
# SUM(total) adds up the 'total' column for all rows in each group
# ROUND(value, 2) rounds to 2 decimal places — important for currency to avoid floating-point noise
# JOIN links two tables on a shared column — here we connect calls.dnis = sources.dnis

print("=== Calls Per Campaign ===")
print(f"{'Campaign':<26} {'Client':<20} {'Channel':<12} {'Calls':>6}")
print("-" * 66)

results = cursor.execute("""
    SELECT s.campaign_name, s.client_name, s.channel, COUNT(*) as call_count
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis   -- join on the shared column (DNIS = phone number dialed)
    GROUP BY s.campaign_name, s.client_name, s.channel
    ORDER BY call_count DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:<20} {row[2]:<12} {row[3]:>6}")

print("\n")
print("=== Revenue Per Campaign ===")
print(f"{'Campaign':<26} {'Channel':<12} {'Orders':>7} {'Revenue':>10}")
print("-" * 57)

results = cursor.execute("""
    SELECT s.campaign_name, s.channel,
           COUNT(o.order_id) as order_count,
           ROUND(SUM(o.total), 2) as total_revenue
    FROM orders o
    JOIN calls c ON o.call_id = c.call_id     -- orders to calls: get the call that created each order
    JOIN sources s ON c.dnis = s.dnis          -- calls to sources: get the campaign name for each call
    GROUP BY s.campaign_name, s.channel
    ORDER BY total_revenue DESC
""").fetchall()

for row in results:
    print(f"{row[0]:<26} {row[1]:<12} {row[2]:>7} ${row[3]:>9.2f}")

# You should see: 4 campaigns with revenue — 2 campaigns had no completed orders yet (VA programs just launched)
# WHY not all 6 campaigns? Two campaigns (Acme Rewards, Summit Credit Card) had no orders in this dataset
print("\nNotice: We JOINed 3 tables — orders to calls to sources.")
print("The JOIN chain follows the foreign keys in our schema diagram.")
print("The channel column lets you compare live agent vs VA performance.")


---

### Why JOINs Instead of One Big Table?

A natural beginner question: *why not just put campaign_name, client_name, and call info all in one table?*

**The "one big table" approach (denormalized):**
```
call_id | dnis        | campaign_name    | client_name | subtotal | payment_status
--------|-------------|------------------|-------------|----------|---------------
call_001| 8005551234  | Acme Spring Sale | Acme Corp   | 69.99    | approved
call_002| 8005551234  | Acme Spring Sale | Acme Corp   | NULL     | NULL
```

Problems with this approach:

| Problem | What Happens |
|---------|-------------|
| **Update anomaly** | Acme renames their campaign. You update 1 row, forget 200 others. Now 3 different campaign names exist for the same phone number. |
| **Insert anomaly** | You cannot add a new campaign (DNIS) until a call comes in — the campaign data has no home without a call row. |
| **Storage waste** | "Acme Spring Sale" + "Acme Corp" stored in every single call row. At 100,000 calls, that is 100,000 redundant copies. |
| **NULL pollution** | Dropped calls have no order. Every dropped call row needs NULL for subtotal, tax, shipping, total, payment_status. |

**The normalized approach (what we built):**
- Campaign name lives in `sources` exactly once
- If Acme renames a campaign, you update 1 row — all queries instantly reflect it
- Dropped calls simply have no row in `orders` — no NULLs needed in the calls table

> **DE rule of thumb:** Normalize to store data once. Denormalize only when query performance demands it (and document why). dbt models often "flatten" normalized tables into wide reporting tables — but the source stays clean.

---

### Exercise 3: JOINs — Connecting Tables

**Task:** Which calls resulted in orders? Which didn't?

Key SQL concepts:
- `INNER JOIN` returns only rows that match in both tables
- `LEFT JOIN` returns all rows from the left table, even if there's no match on the right
- When the right side is `NULL` after a LEFT JOIN, it means no match was found

> **This LEFT JOIN pattern is how you find data quality issues — orphaned records, missing links, broken joins.**

---

### Try It Yourself — Exercise 3

Think before you write:
- INNER JOIN keeps only rows that match in both tables. What will disappear from the result?
- LEFT JOIN keeps all rows from the LEFT table. Which table should be on the left?
- When there is no match on the right side, what value will those columns show?

Write the LEFT JOIN query yourself first:
```sql
SELECT c.call_id, c.channel, c.disposition, o.order_id
FROM calls c
LEFT JOIN orders o ON _____
WHERE _____
```

**Bonus challenge:** Modify the query to show only dropped calls that have no order. Does every dropped call have no order?

In [ ]:
# Exercise 3: JOINs — INNER JOIN vs LEFT JOIN
# WHY JOINs matter: Real data lives across multiple tables. JOINs are how you reunite it.
# INNER JOIN: "Give me rows that exist in BOTH tables" — Venn diagram intersection
# LEFT JOIN:  "Give me ALL rows from the left table, attach right-table data where available"
#             When there is no match on the right side, those columns show NULL
# BEGINNER NOTE: Think of LEFT JOIN like a class roster with optional exam scores
#   Students who skipped the exam still appear — their score column shows NULL

# Part A: INNER JOIN — calls that DID result in orders
print("=== Part A: Calls WITH Orders (INNER JOIN) ===")
print(f"{'Call ID':<12} {'Channel':<12} {'Order ID':<10} {'Total':>8} {'Payment':>10}")
print("-" * 54)

results = cursor.execute("""
    SELECT c.call_id, c.channel, o.order_id, o.total, p.status
    FROM calls c
    INNER JOIN orders o ON c.call_id = o.call_id       -- only calls that have a matching order
    INNER JOIN payments p ON o.order_id = p.order_id   -- only orders that have a matching payment
    ORDER BY c.call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<10} ${row[3]:>7.2f} {row[4]:>10}")
# You should see: 15 rows — exactly the 15 calls that converted to orders
print(f"\nCalls with orders: {len(results)}")

# Part B: LEFT JOIN — find calls WITHOUT orders
# Pattern: LEFT JOIN + WHERE right_side IS NULL = "find what is missing"
# This gap analysis pattern is used constantly in data quality work
print("\n=== Part B: Calls WITHOUT Orders (LEFT JOIN + WHERE NULL) ===")
print(f"{'Call ID':<12} {'Channel':<12} {'Disposition':<14} {'Order?':<8}")
print("-" * 46)

results = cursor.execute("""
    SELECT c.call_id, c.channel, c.disposition, o.order_id
    FROM calls c
    LEFT JOIN orders o ON c.call_id = o.call_id   -- keep ALL calls; attach order if one exists
    WHERE o.order_id IS NULL                       -- keep only calls with NO matching order
    ORDER BY c.call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<14} {'(none)':<8}")

total_calls = cursor.execute("SELECT COUNT(*) FROM calls").fetchone()[0]
# You should see: 25 calls with no order — mostly dropped and transferred, as expected
print(f"\nCalls without orders: {len(results)} out of {total_calls} ({len(results)/total_calls*100:.0f}%)")
# WHY IS NULL not = NULL? In SQL, NULL means "unknown" — it is not equal to anything, even itself
# NULL = NULL evaluates to NULL (not TRUE). NULL IS NULL evaluates to TRUE. Memorize this.
print("\nLEFT JOIN + WHERE IS NULL = 'find what is missing'")
print("This is one of the most important patterns in data quality checking.")


---

### SQL Gotchas — Common Mistakes That Cost Real Engineers Real Time

**Gotcha 1: NULL behavior in comparisons**
```sql
-- These two queries look similar but both miss NULLs:
SELECT * FROM calls WHERE sentiment = 'negative'     -- misses NULL sentiment rows
SELECT * FROM calls WHERE sentiment != 'positive'    -- also misses NULL sentiment rows

-- NULL means "unknown" — it is not equal to anything, not even another NULL
-- To include NULLs: WHERE sentiment != 'positive' OR sentiment IS NULL
```

**Gotcha 2: COUNT(*) vs COUNT(column)**
```sql
SELECT COUNT(*)         FROM calls   -- counts all 40 rows including NULLs
SELECT COUNT(sentiment) FROM calls   -- counts only rows where sentiment is NOT NULL

-- In our data, some dropped calls have NULL sentiment — COUNT(sentiment) < COUNT(*)
-- Rule of thumb: COUNT(*) for "how many rows total", COUNT(col) for "how many have a value"
```

**Gotcha 3: GROUP BY with NULLs**
```sql
SELECT sentiment, COUNT(*) FROM calls GROUP BY sentiment
-- You will see a row where sentiment IS NULL — those are calls with no sentiment score
-- NULLs form their own group in GROUP BY. This is correct behavior, but surprises beginners.
```

**Gotcha 4: SQL has a fixed order of execution**

Regardless of how you write the query, SQL always processes clauses in this order:
```
FROM -> JOIN -> WHERE -> GROUP BY -> HAVING -> SELECT -> ORDER BY -> LIMIT
```
Why it matters:
```sql
-- This FAILS — WHERE runs before SELECT, so the alias 'cnt' does not exist yet
SELECT campaign_name, COUNT(*) as cnt FROM calls WHERE cnt > 5 GROUP BY campaign_name  -- ERROR

-- This WORKS — HAVING filters after GROUP BY, where the alias already exists
SELECT campaign_name, COUNT(*) as cnt FROM calls GROUP BY campaign_name HAVING cnt > 5
```

**Gotcha 5: Date string comparison pitfalls**
```sql
-- ISO 8601 format (YYYY-MM-DDTHH:MM:SSZ) sorts correctly as a plain string
WHERE call_started_at > '2026-03-10'    -- works (ISO dates sort correctly as strings)
WHERE call_started_at > '3/10/2026'     -- BREAKS — '3' sorts after '2026' alphabetically

-- Always store and compare dates in ISO 8601 format: YYYY-MM-DDTHH:MM:SSZ
```

> **Interview tip:** When asked about NULL behavior — "NULL means unknown. Any comparison to NULL returns NULL, not TRUE or FALSE. Always use IS NULL and IS NOT NULL, never = NULL or != NULL."

---

### Exercise 4: The UTC Bug — Date Functions

**Task:** Why do evening calls show up on the wrong date?

This is a **real production bug.** Reports showed inflated Monday numbers and deflated Friday numbers because UTC-to-EST conversion wasn't applied. Evening calls at 7 PM EST = 12 AM UTC the next day.

In our data, calls 021-025 happened on the evening of March 10 EST, but their UTC timestamps say March 11.

> **You'll fix this properly in M03 (SQL) and again in M06 (PySpark).** This is the kind of bug that costs businesses real money — wrong daily totals, wrong billing, wrong campaign attribution.

---

### Try It Yourself — Exercise 4

Before running the solution, predict the answer:
- We have 40 calls total
- We inserted calls on both March 10 and March 11 UTC
- But 5 evening EST calls cross the UTC date boundary

When you run `GROUP BY DATE(call_started_at)` (UTC), what counts do you expect for each date?
When you run `GROUP BY DATE(call_started_at, '-5 hours')` (EST), how do you expect the counts to shift?

Write your prediction down before running. If it is wrong, that gap is the learning.

In [ ]:
# Exercise 4: The UTC Bug
# WHY this matters: Wrong timestamps = wrong daily totals = wrong business decisions
# UTC (Coordinated Universal Time) is the global standard — databases store it to avoid timezone confusion
# BUT reports must show LOCAL time. Forget to convert and evening calls appear on the WRONG day.
# EST = UTC - 5 hours. A call at 7 PM EST = midnight UTC = rolls to the NEXT calendar day.

# Step 1: Count calls per day using UTC date (the WRONG way for business reporting)
# DATE() in SQLite extracts the calendar date portion from a timestamp string
print("=== Calls Per Day — Using UTC Date (WRONG) ===")
results_utc = cursor.execute("""
    SELECT DATE(call_started_at) as call_date_utc,
           COUNT(*) as call_count
    FROM calls
    GROUP BY DATE(call_started_at)
    ORDER BY call_date_utc
""").fetchall()

for row in results_utc:
    print(f"  {row[0]}: {row[1]} calls")

# Step 2: Count calls per day using EST date (the CORRECT way for client reporting)
# SQLite modifier syntax: '-5 hours' shifts the timestamp back 5 hours before extracting the date
# In BigQuery: DATETIME(ts, 'America/New_York') — handles Daylight Saving Time automatically
print("\n=== Calls Per Day — Using EST Date (CORRECT) ===")
results_est = cursor.execute("""
    SELECT DATE(call_started_at, '-5 hours') as call_date_est,
           COUNT(*) as call_count
    FROM calls
    GROUP BY DATE(call_started_at, '-5 hours')
    ORDER BY call_date_est
""").fetchall()

for row in results_est:
    print(f"  {row[0]}: {row[1]} calls")

# Step 3: Isolate the problem calls — rows where UTC date != EST date
# These are the "boundary crossers" — evening EST calls that roll to the next UTC calendar day
print("\n=== The Problem Calls (evening EST to next day UTC) ===")
print(f"{'Call ID':<12} {'Channel':<12} {'UTC Timestamp':<24} {'UTC Date':<12} {'EST Date':<12}")
print("-" * 72)

results = cursor.execute("""
    SELECT call_id, channel,
           call_started_at,
           DATE(call_started_at) as utc_date,
           DATE(call_started_at, '-5 hours') as est_date
    FROM calls
    WHERE DATE(call_started_at) != DATE(call_started_at, '-5 hours')
    ORDER BY call_started_at
""").fetchall()

for row in results:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:<24} {row[3]:<12} {row[4]:<12} <- date changes!")

# You should see: 5 calls (call_021 through call_025) — the intentional UTC bug calls we inserted
print(f"\n{len(results)} calls appear on the wrong date when using UTC.")
print("If you report daily totals without converting to EST, your numbers are WRONG.")
print("\nThis exact bug has caused:")
print("  - Inflated Monday numbers (Sunday evening UTC = Monday)")
print("  - Deflated Friday numbers (Friday evening EST = Saturday UTC)")
print("  - Wrong billing calculations for daily/monthly client invoices")


---

### Why Is the Timezone Bug So Dangerous? (Business Impact)

This is not just an academic exercise. Here is what happens in production when you skip timezone conversion:

**Scenario: Daily Revenue Report**

Your client expects a daily revenue report every morning at 8 AM EST. You write:
```sql
SELECT DATE(call_started_at) as call_date, SUM(o.total) as revenue
FROM calls c JOIN orders o ON c.call_id = o.call_id
GROUP BY DATE(call_started_at)
```

With our 40-call dataset, the UTC bug misassigns 5 evening calls from March 10 to March 11.

| What the report shows | What actually happened |
|-----------------------|------------------------|
| March 10: fewer calls, lower revenue | March 10: 5 more calls, more revenue |
| March 11: more calls, higher revenue | March 11: 5 fewer calls, less revenue |

**Real consequences:**

1. **Wrong billing** — If the client is billed per call per day, the March 10 invoice is too low and March 11 is too high. Finance reconciliation becomes a weekly headache.
2. **Wrong campaign attribution** — Evening campaigns with high conversion rates appear to belong to the next morning's traffic. Campaign managers optimize on bad data.
3. **Wrong Monday numbers** — Sunday evening EST = Monday UTC. Every Monday looks artificially inflated. Teams investigate a phantom spike weekly.
4. **Compound error** — Daily to weekly to monthly rollups all inherit the same error. A month-end report can be off by hundreds of thousands of dollars at high call volume.

**The fix across the stack:**
- SQLite (learning): `DATE(call_started_at, '-5 hours')`
- BigQuery (production): `DATETIME(call_started_at, 'America/New_York')` — handles DST automatically
- PySpark (M06): `from_utc_timestamp(col('call_started_at'), 'America/New_York')`

> **Note:** We hardcoded `-5 hours` (EST) for simplicity. In production, always use the named timezone `America/New_York` — it handles the spring/fall Daylight Saving Time clock change automatically. Missing DST = a 1-hour error twice a year on every daily report.

You will implement the proper fix in M03 (Advanced SQL) and again in M06 (PySpark).

---

## 7. Key Takeaways

1. **Project structure and `.gitignore` are Day 1 setup — not optional.** Data files, credentials, and generated artifacts stay out of Git.
2. **Git branching keeps your pipeline code safe.** Feature branches → pull request → code review → merge. Never push directly to main.
3. **Command-line tools give you instant data profiling.** `head`, `wc -l`, `grep`, `cut | sort | uniq -c` — faster than loading Python for a quick check.
4. **SQL is the primary DE language — every tool speaks it.** BigQuery, PySpark, Hive, Airflow, dbt — all use SQL or a SQL dialect.
5. **The call center schema (5 tables) is the dataset for the entire program.** `calls` → `orders` → `payments`, with `sources` and `clients` as reference tables.
6. **The platform is primarily live agent, with VA as a newer channel.** The `channel` column lets you compare performance across both.
7. **JOINs connect the story:** calls → orders → payments → campaigns. Follow the foreign keys.
8. **LEFT JOINs find what's missing** — a core data quality pattern. "Show me all calls that did NOT result in an order."
9. **UTC timestamps are a real production bug — always convert for reporting.** Evening EST calls appear as the next day in UTC. This affects daily totals, billing, and campaign attribution.

---

## 8. Homework

### 1. Initialize Your Project Repo

Run these commands in your terminal:

```bash
mkdir de-call-center-analytics
cd de-call-center-analytics
git init

# Create folder structure
mkdir -p data/raw data/processed notebooks src tests dags

# Create .gitignore (use the one from Section 1)
# Create README.md with project name and your name

# First commit
git add .gitignore README.md
git commit -m "feat: initialize project structure"
```

**Deliverable:** Git repo with project structure and initial commit.

### 2. SQLBolt Lessons 7-12 (30 min)

Complete [SQLBolt](https://sqlbolt.com/) Lessons 7-12 — JOINs, aggregations, order of execution, subqueries.

### 3. Challenge Query

Write a query that finds the **top 3 campaigns by conversion rate** (calls that resulted in orders / total calls per campaign). Bring your query to the next session.

Hint: You'll need `calls`, `orders`, and `sources`. Think about which JOIN type to use.

### 4. Preview M03

Skim what **window functions** are — just search "SQL window functions" and read for 10 minutes. You don't need to master them yet; just know they exist.

---

**Next session:** M03 — Advanced SQL for Data Engineering (window functions, CTEs, optimization)